# Detectra AI — Fusion Engine Training
**CrossModalTransformer:** d_model=256, 8-head cross-attention, 2 layers
**GPU:** Google Colab T4 (~4 hours)
**Output:** fusion_transformer.pt → copy to backend/models/


In [ ]:
!pip install -q torch transformers
import torch,math,json,time,shutil
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA: {torch.cuda.is_available()} | Device: {DEVICE}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR=Path("/content/drive/MyDrive/detectra_models")
SAVE_DIR.mkdir(parents=True,exist_ok=True)
print(f"Save dir: {SAVE_DIR}")

In [ ]:
# Architecture constants — must match fusion_engine.py
VISUAL_FEAT_DIM=213  # 80 COCO + 32 logo + 101 UCF-101
AUDIO_FEAT_DIM=12    # 4 speech + 8 audio event
D_MODEL=256; N_HEADS=8; N_CROSS=2
SCENE_LABELS=["sports_event","music_performance","news_broadcast","advertisement",
    "documentary","conversation","action_sequence","nature_wildlife",
    "surveillance","tutorial","unknown"]
N_SCENES=len(SCENE_LABELS)
print(f"V_DIM={VISUAL_FEAT_DIM} A_DIM={AUDIO_FEAT_DIM} Scenes={N_SCENES}")

In [ ]:
class TPE(nn.Module):
    def __init__(self,d,mx=3600):
        super().__init__()
        pe=torch.zeros(mx,d)
        pos=torch.arange(mx,dtype=torch.float).unsqueeze(1)
        div=torch.exp(torch.arange(0,d,2).float()*(-math.log(10000.0)/d))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class CrossModalTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.ve=nn.Sequential(nn.Linear(VISUAL_FEAT_DIM,D_MODEL),nn.LayerNorm(D_MODEL),nn.ReLU())
        self.ae=nn.Sequential(nn.Linear(AUDIO_FEAT_DIM,D_MODEL),nn.LayerNorm(D_MODEL),nn.ReLU())
        self.pe=TPE(D_MODEL)
        self.ca=nn.ModuleList([nn.MultiheadAttention(D_MODEL,N_HEADS,dropout=0.1,batch_first=True) for _ in range(N_CROSS)])
        self.cn=nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(N_CROSS)])
        enc=nn.TransformerEncoderLayer(D_MODEL,N_HEADS,512,0.1,batch_first=True,norm_first=True)
        self.te=nn.TransformerEncoder(enc,2)
        self.sh=nn.Linear(D_MODEL,N_SCENES)
        self.ah=nn.Sequential(nn.Linear(D_MODEL,64),nn.ReLU(),nn.Linear(64,1),nn.Sigmoid())
        self.ch=nn.Sequential(nn.Linear(D_MODEL*2,64),nn.ReLU(),nn.Linear(64,1),nn.Sigmoid())
    def forward(self,v,a):
        v=self.pe(self.ve(v)); a=self.pe(self.ae(a))
        for att,nm in zip(self.ca,self.cn):
            vat,_=att(v,a,a); v=nm(v+vat)
        f=self.te(v+a)
        return self.sh(f),self.ah(f).squeeze(-1),self.ch(torch.cat([v,a],-1)).squeeze(-1)

model=CrossModalTransformer().to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Synthetic training data
# For real training: load pre-extracted feature vectors from your individual module outputs
N,T=1000,60
v=torch.rand(N,T,VISUAL_FEAT_DIM); a=torch.rand(N,T,AUDIO_FEAT_DIM)
sc=torch.randint(0,N_SCENES,(N,T)); anom=torch.rand(N,T); corr=torch.rand(N,T)
tr_ds=TensorDataset(v[:800],a[:800],sc[:800],anom[:800],corr[:800])
vl_ds=TensorDataset(v[800:],a[800:],sc[800:],anom[800:],corr[800:])
tr_ld=DataLoader(tr_ds,16,shuffle=True); vl_ld=DataLoader(vl_ds,16)
print(f"Train:{len(tr_ds)} Val:{len(vl_ds)}")

In [ ]:
EPOCHS=30
opt=AdamW(model.parameters(),lr=3e-4,weight_decay=0.01)
sch=CosineAnnealingLR(opt,EPOCHS)
ce_fn=nn.CrossEntropyLoss(); mse_fn=nn.MSELoss()
best=float("inf")
for ep in range(EPOCHS):
    model.train(); tl=0
    for vb,ab,sb,anb,cb in tr_ld:
        vb,ab,sb,anb,cb=[x.to(DEVICE) for x in [vb,ab,sb,anb,cb]]
        opt.zero_grad()
        sp,ap,cp=model(vb,ab)
        loss=ce_fn(sp.reshape(-1,N_SCENES),sb.reshape(-1))+0.5*mse_fn(ap,anb)+0.5*mse_fn(cp,cb)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); tl+=loss.item()
    sch.step()
    model.eval(); vl2=0
    with torch.no_grad():
        for vb,ab,sb,anb,cb in vl_ld:
            vb,ab,sb,anb,cb=[x.to(DEVICE) for x in [vb,ab,sb,anb,cb]]
            sp,ap,cp=model(vb,ab)
            vl2+=ce_fn(sp.reshape(-1,N_SCENES),sb.reshape(-1)).item()+0.5*mse_fn(ap,anb).item()+0.5*mse_fn(cp,cb).item()
    avg_v=vl2/len(vl_ld)
    if avg_v<best: best=avg_v; torch.save(model.state_dict(),"/content/fusion_best.pt")
    if (ep+1)%5==0: print(f"Ep {ep+1:02d}/{EPOCHS} | train={tl/len(tr_ld):.4f} | val={avg_v:.4f}")
print(f"Best val loss: {best:.4f}")

In [ ]:
shutil.copy("/content/fusion_best.pt", SAVE_DIR/"fusion_transformer.pt")
print(f"Saved: {SAVE_DIR}/fusion_transformer.pt")
print(">>> Copy to: backend/models/fusion_transformer.pt")
with open(SAVE_DIR/"fusion_config.json","w") as f:
    json.dump({"VISUAL_FEAT_DIM":VISUAL_FEAT_DIM,"AUDIO_FEAT_DIM":AUDIO_FEAT_DIM,
        "D_MODEL":D_MODEL,"N_HEADS":N_HEADS,"SCENE_LABELS":SCENE_LABELS},f,indent=2)
print("Config saved.")